# Módulo 1 — Word Embeddings

Primeiro módulo do curso intermediário de NLP. A ideia aqui é treinar nossos
próprios **word embeddings** do zero, sem baixar nada da internet, usando um
mini Word2Vec (skip-gram) implementado em PyTorch sobre uma base de SMS
spam/ham criada localmente (mesmo estilo do curso introdutório, mas uma base
nova e independente).

No final, salvamos o texto limpo, o vocabulário e os embeddings treinados —
os próximos módulos (Preparing Text Data, RNN, LSTM, Bidirectional LSTM) vão
reaproveitar tudo isso.

Tente resolver antes de olhar `exercicios_1_word_embeddings_solucoes.ipynb`.

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

sms_data = [
    ("ham", "Hey, are we still on for lunch tomorrow?"),
    ("ham", "I'll call you when I get home from work."),
    ("ham", "Can you send me the notes from today's class?"),
    ("ham", "Happy birthday! Hope you have an amazing day."),
    ("ham", "Running a bit late, be there in 10 minutes."),
    ("ham", "Thanks for the ride yesterday, really appreciated."),
    ("ham", "Don't forget the meeting tomorrow morning."),
    ("ham", "Can we reschedule our meeting to tomorrow?"),
    ("ham", "I'll be at the meeting, see you tomorrow."),
    ("ham", "Thanks so much, talk to you tomorrow."),
    ("ham", "Let's grab lunch after the meeting tomorrow."),
    ("ham", "Sorry I missed your call earlier, call me back."),
    ("ham", "Can you call me when you get a chance?"),
    ("ham", "I'll call the doctor to book an appointment."),
    ("ham", "Mom said dinner is ready, come home now."),
    ("ham", "See you at the gym later tonight."),
    ("ham", "Traffic is bad, I might be late for work."),
    ("ham", "Can you pick up the kids from school today?"),
    ("ham", "I forgot my notes at home, can you scan them?"),
    ("ham", "Movie night this weekend? Let me know."),
    ("ham", "Coffee tomorrow morning before work?"),
    ("ham", "Thanks for helping me with the project today."),
    ("ham", "The project meeting got moved to tomorrow."),
    ("ham", "Good night, talk to you tomorrow."),
    ("ham", "Congrats on the new job, so happy for you."),
    ("ham", "Can we do groceries together this weekend?"),
    ("ham", "I'll be home late tonight, don't wait for dinner."),
    ("ham", "Thanks again for everything, means a lot."),
    ("ham", "Let's plan the weekend trip, call me tonight."),
    ("ham", "Dad wants to know if you're coming home tomorrow."),
    ("ham", "Class got cancelled, no notes needed today."),
    ("ham", "I'll bring the notes to the meeting tomorrow."),
    ("ham", "Happy to help, just call me anytime."),
    ("ham", "See you tomorrow at the usual coffee place."),
    ("ham", "The doctor's appointment is confirmed for tomorrow."),
    ("ham", "Sorry for the late reply, was in a meeting."),
    ("ham", "Can you send the project file before tomorrow?"),
    ("ham", "Thanks for lunch, let's do it again soon."),
    ("ham", "I'll pick you up for the gym tomorrow morning."),
    ("ham", "Meeting notes are attached, check before tomorrow."),
    ("ham", "Happy weekend! See you at the gym."),
    ("ham", "Call me back when you're free, nothing urgent."),
    ("ham", "Thanks for the birthday wishes, means a lot."),
    ("ham", "Let's catch up over coffee this weekend."),
    ("ham", "I'm at work, will call you after the meeting."),
    ("ham", "Can you check the notes and call me tonight?"),
    ("ham", "Dinner at mom's tomorrow, don't be late."),
    ("ham", "Thanks for covering my shift today."),
    ("ham", "See you tomorrow, drive safe."),
    ("ham", "The meeting tomorrow is confirmed for 10am."),
    ("ham", "Congrats again, the whole team is proud."),
    ("ham", "Can we push the call to tomorrow afternoon?"),
    ("ham", "I'll send the notes right after the meeting."),
    ("ham", "Thanks for the coffee this morning."),
    ("ham", "Let's meet tomorrow to finish the project."),
    ("ham", "Good luck with the appointment tomorrow."),
    ("ham", "Call me tomorrow, I have some news to share."),
    ("ham", "Thanks for picking up the kids today."),
    ("ham", "See you at the meeting, bring your notes."),
    ("ham", "Home now, dinner will be ready soon."),
    ("ham", "Can't wait for the weekend trip, thanks for planning."),
    ("ham", "Sorry, stuck in traffic, call you when I'm home."),
    ("spam", "URGENT! You have won a free prize, claim now!"),
    ("spam", "Congratulations! You've been selected for a free cash award."),
    ("spam", "WIN a guaranteed cash prize, text WIN to claim now."),
    ("spam", "URGENT! Your mobile number has won a free prize, call now."),
    ("spam", "Free entry to win a cash prize, click the link now."),
    ("spam", "Claim your free cash prize now, urgent reply needed."),
    ("spam", "You have been selected to win a free voucher, claim now."),
    ("spam", "URGENT! Reply now to claim your free cash prize."),
    ("spam", "Winner! You've won a free cash award, text CLAIM now."),
    ("spam", "Free cash prize waiting, call now to claim urgent offer."),
    ("spam", "Exclusive offer: claim your free prize now, urgent!"),
    ("spam", "URGENT! Limited time, claim your free cash now."),
    ("spam", "Congratulations winner! Free cash prize, reply now to claim."),
    ("spam", "Your account has won a free prize, click now to claim."),
    ("spam", "Text WIN now for a chance to claim free cash prize."),
    ("spam", "URGENT offer! Free prize guaranteed, call now to claim."),
    ("spam", "You are a winner! Claim your free cash prize urgent."),
    ("spam", "Free voucher waiting, urgent reply to claim cash prize."),
    ("spam", "Congratulations! Urgent, claim your guaranteed prize now."),
    ("spam", "WIN free cash now, click link, urgent claim required."),
    ("spam", "URGENT! You have a free prize, text CLAIM to collect now."),
    ("spam", "Selected winner, free cash prize, call urgent now."),
    ("spam", "Claim now! Free prize and cash bonus, urgent offer."),
    ("spam", "URGENT! Free cash award waiting, reply CLAIM now."),
    ("spam", "Congratulations, you win! Claim your free prize urgent now."),
    ("ham", "Are you free tonight for dinner?"),
    ("ham", "Congrats on the win, let's celebrate this weekend!"),
    ("ham", "I'll text you the address now."),
    ("ham", "Call me back urgent, mom needs you home."),
    ("ham", "Can I get your number to text you later?"),
    ("ham", "Free tomorrow afternoon? Let's catch up."),
    ("ham", "Great news, call me now, I'm so excited!"),
    ("spam", "Reply now to secure your exclusive reward before it expires."),
    ("spam", "Your account is due a bonus, confirm today to receive it."),
    ("spam", "Act fast, this offer expires today, don't miss out."),
    ("spam", "You are eligible for a special gift, respond immediately."),
    ("spam", "Final notice: your reward is ready, confirm to collect."),
    ("spam", "Selected for an exclusive deal, confirm today to receive."),
]

sms = pd.DataFrame(sms_data, columns=["label", "text"])
sms.head()

## 1.1 Explorar os dados

Quantas mensagens temos? Qual a distribuição de `label` (`value_counts`)?

In [ ]:
# 1.1


## 1.2 Limpar o texto

Crie `clean_text`: deixe o texto em minúsculas e remova pontuação/números
usando uma regex (mantenha apenas letras e espaços).

In [ ]:
# 1.2


## 1.3 Tokenizar e construir o vocabulário

1. Crie `tokens`: `clean_text` dividido em palavras (`.str.split()`).
2. Junte todos os tokens de todas as mensagens e pegue as palavras únicas.
3. Construa um dicionário `word2idx` reservando os índices especiais:
   - `0` para `<PAD>` (preenchimento, usado no Módulo 2)
   - `1` para `<UNK>` (palavra desconhecida)
   - a partir de `2`, uma palavra única por índice.
4. Guarde `vocab_size = len(word2idx)`.

In [ ]:
# 1.3


## 1.4 Gerar pares skip-gram

O skip-gram tenta prever palavras de **contexto** a partir de uma palavra
**alvo**. Para cada mensagem (lista de tokens), para cada posição `i`
(palavra alvo), colete as palavras dentro de uma janela (`window = 2`) ao
redor dela como contexto, e gere pares `(idx_alvo, idx_contexto)` usando o
`word2idx`.

Guarde todos os pares em uma lista `pairs` (lista de tuplas de inteiros).

In [ ]:
# 1.4


## 1.5 Treinar o mini Word2Vec (skip-gram) em PyTorch

1. Converta `pairs` em dois tensores `targets` e `contexts` (`torch.long`).
2. Defina `embed_dim = 32`.
3. Crie um modelo simples:
   - `embedding = nn.Embedding(vocab_size, embed_dim)`
   - `linear = nn.Linear(embed_dim, vocab_size)`
   - `forward(x)`: `linear(embedding(x))` (logits sobre o vocabulário)
4. Treine com `nn.CrossEntropyLoss()` e `torch.optim.Adam(lr=0.01)` por
   `200` épocas, imprimindo a loss a cada 50 épocas.

In [ ]:
# 1.5


## 1.6 Explorar os embeddings treinados

1. Extraia a matriz de embeddings treinada (`embedding.weight.detach().numpy()`).
2. Escreva uma função `most_similar(word, topn=5)` que calcule a
   similaridade de cosseno entre o embedding de `word` e todas as outras
   palavras do vocabulário, retornando as `topn` mais parecidas.
3. Teste com `most_similar("free")` e `most_similar("meeting")` — as
   palavras mais próximas fazem sentido (mais spam perto de "free", mais
   ham perto de "meeting")?

In [ ]:
# 1.6


## 1.7 Salvar para o próximo módulo

Salve:
- `sms[["label", "text", "clean_text"]]` em `sms_clean.csv` (`index=False`)
- `word2idx` em `vocab.json` (`json.dump`)
- a matriz de embeddings treinada em `embedding_matrix.npy` (`np.save`)

In [ ]:
# 1.7
